In [18]:
import os
from dotenv import load_dotenv
from dbrepo.RestClient import RestClient

load_dotenv("../.env")

ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
USERNAME = os.getenv("DBREPO_USERNAME")
PASSWORD = os.getenv("DBREPO_PASSWORD")
DATABASE_ID = os.getenv("DBREPO_DATABASE_ID")

client = RestClient(
    endpoint=ENDPOINT,
    username=USERNAME,
    password=PASSWORD
)

print("Current user:", client.whoami())
print("Database ID:", DATABASE_ID)

federicoa88
Current user: federicoa88
Database ID: bfa4385b-54a9-4ae3-b4f4-cb503d7bb016


# WP2/T2.1 — DBRepo schema design and table publication

This notebook implements the WP2/T2.1 database infrastructure task for the precipitation chemistry experiment.

The goal is to transform the original CSV input data into a normalized relational schema and publish the resulting tables in DBRepo using the REST API.

The notebook covers:
1. loading and inspecting the original CSV files,
2. creating normalized relational tables,
3. generating SQL `CREATE TABLE` statements,
4. generating an ER diagram,
5. uploading the tables to DBRepo with source DOI and license metadata,
6. verifying the created DBRepo tables.

The source dataset is **“Concentrations of major ions in wet precipitation samples in Austria”** by Redl, Steinkogler and Kasper-Giebl, available through the TU Wien Research Data Repository.

## 1. Load source data

The experiment uses two original CSV files:

- `precipitationdata.csv`: precipitation chemistry measurements and quality flags
- `stationcoordinates.csv`: station codes and geographic coordinates

These files are loaded from the local `data/` folder.

In [3]:
import pandas as pd

precipitation_path = "../data/precipitationdata.csv"
stations_path = "../data/stationcoordinates.csv"

precipitation_df = pd.read_csv(precipitation_path)
stations_df = pd.read_csv(stations_path)

print("Precipitation shape:", precipitation_df.shape)
print("Stations shape:", stations_df.shape)

Precipitation shape: (8572, 28)
Stations shape: (14, 3)


In [4]:
precipitation_df.head()

,Datum,Ort,NS,NS_flag,LF,LF_flag,pH,pH_flag,NH4,NH4_flag,...,Cl,Cl_flag,NO3,NO3_flag,SO4,SO4_flag,Pb,Pb_flag,Cd,Cd_flag
0,2014-10-01,SO,6.700000,1.0,4.05,1.0,5.52,1.0,0.158,1.0,...,0.016,1.0,0.493,1.0,0.573,1.0,0.00017,1.0,0.032874,1.0
1,2014-10-01,AF,1.400000,1.0,16.95,1.0,5.72,1.0,1.187,1.0,...,0.105,1.0,4.417,1.0,1.424,1.0,NaN,7.0,NaN,7.0
2,2014-10-01,LI,3.800000,1.0,NaN,7.0,NaN,7.0,NaN,7.0,...,NaN,7.0,NaN,7.0,NaN,7.0,NaN,7.0,NaN,7.0
3,2014-10-01,LU,11.500000,1.0,13.47,1.0,6.34,1.0,0.381,1.0,...,0.038,1.0,1.788,1.0,0.876,1.0,NaN,7.0,NaN,7.0
4,2014-10-01,IV,2.599653,1.0,4.40,1.0,5.77,1.0,0.376,1.0,...,0.050,1.0,0.780,1.0,0.240,1.0,NaN,7.0,NaN,7.0


In [5]:
stations_df.head()

,Ort,Lat,Long
0,HF,47.470833,10.680833
1,NB,47.662222,12.226944
2,IV,46.818056,12.351667
3,NH,47.956389,13.016667
4,WW,47.421667,13.253333


## 2. Inspect source data

Before designing the relational schema, the source columns, data types, missing values, and station identifiers are inspected.

In [6]:
print("Precipitation columns:")
print(precipitation_df.columns.tolist())

print("\nStation columns:")
print(stations_df.columns.tolist())

Precipitation columns:
['Datum', 'Ort', 'NS', 'NS_flag', 'LF', 'LF_flag', 'pH', 'pH_flag', 'NH4', 'NH4_flag', 'Na', 'Na_flag', 'K', 'K_flag', 'Ca', 'Ca_flag', 'Mg', 'Mg_flag', 'Cl', 'Cl_flag', 'NO3', 'NO3_flag', 'SO4', 'SO4_flag', 'Pb', 'Pb_flag', 'Cd', 'Cd_flag']

Station columns:
['Ort', 'Lat', 'Long']


In [7]:
precipitation_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8572 entries, 0 to 8571
Data columns (total 28 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Datum     8572 non-null   str    
 1   Ort       8572 non-null   str    
 2   NS        8537 non-null   float64
 3   NS_flag   8572 non-null   float64
 4   LF        7763 non-null   float64
 5   LF_flag   8572 non-null   float64
 6   pH        7748 non-null   float64
 7   pH_flag   8572 non-null   float64
 8   NH4       8062 non-null   float64
 9   NH4_flag  8572 non-null   float64
 10  Na        7922 non-null   float64
 11  Na_flag   8572 non-null   float64
 12  K         7904 non-null   float64
 13  K_flag    8572 non-null   float64
 14  Ca        7928 non-null   float64
 15  Ca_flag   8572 non-null   float64
 16  Mg        7935 non-null   float64
 17  Mg_flag   8572 non-null   float64
 18  Cl        8026 non-null   float64
 19  Cl_flag   8572 non-null   float64
 20  NO3       8054 non-null   float64
 21  NO

In [8]:
stations_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Ort     14 non-null     str    
 1   Lat     14 non-null     float64
 2   Long    14 non-null     float64
dtypes: float64(2), str(1)
memory usage: 468.0 bytes


In [9]:
precipitation_df.isna().sum()

Datum          0
Ort            0
NS            35
NS_flag        0
LF           809
LF_flag        0
pH           824
pH_flag        0
NH4          510
NH4_flag       0
Na           650
Na_flag        0
K            668
K_flag         0
Ca           644
Ca_flag        0
Mg           637
Mg_flag        0
Cl           546
Cl_flag        0
NO3          518
NO3_flag       0
SO4          522
SO4_flag       0
Pb          6922
Pb_flag        0
Cd          6922
Cd_flag        0
dtype: int64

In [10]:
print("Stations in precipitation data:")
print(sorted(precipitation_df["Ort"].dropna().unique()))

print("\nStations in stationcoordinates data:")
print(stations_df.head())

Stations in precipitation data:
['AF', 'DR', 'GS', 'HF', 'HG', 'IV', 'LI', 'LU', 'MB', 'NB', 'NH', 'OS', 'SO', 'WW']

Stations in stationcoordinates data:
  Ort        Lat       Long
0  HF  47.470833  10.680833
1  NB  47.662222  12.226944
2  IV  46.818056  12.351667
3  NH  47.956389  13.016667
4  WW  47.421667  13.253333


## 3. Prepare normalized relational tables

The original CSV structure is transformed into three relational tables:

- `stations`
- `precipitation_measurements`
- `measurement_variables`

This separates station metadata, repeated measurements, and variable/unit descriptions.

### Create stations table

In [11]:
stations_table = stations_df.copy()

stations_table = stations_table.rename(columns={
    "Ort": "station_code",
    "Lat": "latitude",
    "Long": "longitude"
})

stations_table.insert(0, "station_id", range(1, len(stations_table) + 1))

stations_table.head()

,station_id,station_code,latitude,longitude
0,1,HF,47.470833,10.680833
1,2,NB,47.662222,12.226944
2,3,IV,46.818056,12.351667
3,4,NH,47.956389,13.016667
4,5,WW,47.421667,13.253333


### Create precipitation measurements table

In [12]:
precipitation_table = precipitation_df.copy()

precipitation_table = precipitation_table.merge(
    stations_table[["station_id", "station_code"]],
    left_on="Ort",
    right_on="station_code",
    how="left"
)

precipitation_table = precipitation_table.drop(columns=["Ort", "station_code"])

precipitation_table = precipitation_table.rename(columns={
    "Datum": "sample_date"
})

precipitation_table.insert(0, "measurement_id", range(1, len(precipitation_table) + 1))

precipitation_table.head()

,measurement_id,sample_date,NS,NS_flag,LF,LF_flag,pH,pH_flag,NH4,NH4_flag,...,Cl_flag,NO3,NO3_flag,SO4,SO4_flag,Pb,Pb_flag,Cd,Cd_flag,station_id
0,1,2014-10-01,6.700000,1.0,4.05,1.0,5.52,1.0,0.158,1.0,...,1.0,0.493,1.0,0.573,1.0,0.00017,1.0,0.032874,1.0,6
1,2,2014-10-01,1.400000,1.0,16.95,1.0,5.72,1.0,1.187,1.0,...,1.0,4.417,1.0,1.424,1.0,NaN,7.0,NaN,7.0,14
2,3,2014-10-01,3.800000,1.0,NaN,7.0,NaN,7.0,NaN,7.0,...,7.0,NaN,7.0,NaN,7.0,NaN,7.0,NaN,7.0,7
3,4,2014-10-01,11.500000,1.0,13.47,1.0,6.34,1.0,0.381,1.0,...,1.0,1.788,1.0,0.876,1.0,NaN,7.0,NaN,7.0,8
4,5,2014-10-01,2.599653,1.0,4.40,1.0,5.77,1.0,0.376,1.0,...,1.0,0.780,1.0,0.240,1.0,NaN,7.0,NaN,7.0,3


### Reorder precipitation table columns

In [13]:
front_cols = ["measurement_id", "station_id", "sample_date"]

other_cols = [c for c in precipitation_table.columns if c not in front_cols]

precipitation_table = precipitation_table[front_cols + other_cols]

precipitation_table.head()

,measurement_id,station_id,sample_date,NS,NS_flag,LF,LF_flag,pH,pH_flag,NH4,...,Cl,Cl_flag,NO3,NO3_flag,SO4,SO4_flag,Pb,Pb_flag,Cd,Cd_flag
0,1,6,2014-10-01,6.700000,1.0,4.05,1.0,5.52,1.0,0.158,...,0.016,1.0,0.493,1.0,0.573,1.0,0.00017,1.0,0.032874,1.0
1,2,14,2014-10-01,1.400000,1.0,16.95,1.0,5.72,1.0,1.187,...,0.105,1.0,4.417,1.0,1.424,1.0,NaN,7.0,NaN,7.0
2,3,7,2014-10-01,3.800000,1.0,NaN,7.0,NaN,7.0,NaN,...,NaN,7.0,NaN,7.0,NaN,7.0,NaN,7.0,NaN,7.0
3,4,8,2014-10-01,11.500000,1.0,13.47,1.0,6.34,1.0,0.381,...,0.038,1.0,1.788,1.0,0.876,1.0,NaN,7.0,NaN,7.0
4,5,3,2014-10-01,2.599653,1.0,4.40,1.0,5.77,1.0,0.376,...,0.050,1.0,0.780,1.0,0.240,1.0,NaN,7.0,NaN,7.0


### Verify station matching

In [14]:
precipitation_table["station_id"].isna().sum()

np.int64(0)

### Create measurement variable metadata table

In [14]:
measurement_variables = pd.DataFrame([
    {"variable_code": "NS", "label": "Precipitation amount", "unit": "mm"},
    {"variable_code": "LF", "label": "Conductivity", "unit": "µS/cm"},
    {"variable_code": "pH", "label": "pH value", "unit": "dimensionless"},
    {"variable_code": "NH4", "label": "Ammonium concentration", "unit": "mg/L"},
    {"variable_code": "Na", "label": "Sodium concentration", "unit": "mg/L"},
    {"variable_code": "K", "label": "Potassium concentration", "unit": "mg/L"},
    {"variable_code": "Ca", "label": "Calcium concentration", "unit": "mg/L"},
    {"variable_code": "Mg", "label": "Magnesium concentration", "unit": "mg/L"},
    {"variable_code": "Cl", "label": "Chloride concentration", "unit": "mg/L"},
    {"variable_code": "NO3", "label": "Nitrate concentration", "unit": "mg/L"},
    {"variable_code": "SO4", "label": "Sulfate concentration", "unit": "mg/L"},
    {"variable_code": "Pb", "label": "Lead concentration", "unit": "µg/L"},
    {"variable_code": "Cd", "label": "Cadmium concentration", "unit": "µg/L"},
])

measurement_variables.insert(0, "variable_id", range(1, len(measurement_variables) + 1))

measurement_variables

,variable_id,variable_code,label,unit
0,1,NS,Precipitation amount,mm
1,2,LF,Conductivity,µS/cm
2,3,pH,pH value,dimensionless
3,4,NH4,Ammonium concentration,mg/L
4,5,Na,Sodium concentration,mg/L
5,6,K,Potassium concentration,mg/L
6,7,Ca,Calcium concentration,mg/L
7,8,Mg,Magnesium concentration,mg/L
8,9,Cl,Chloride concentration,mg/L
9,10,NO3,Nitrate concentration,mg/L


## 4. Generate SQL schema

The following SQL statements document the relational database schema.

The schema defines primary keys, the foreign-key relationship between stations and precipitation measurements, and a metadata table for measurement variables.

In [16]:
sql_create_statements = """
CREATE TABLE stations (
    station_id INTEGER PRIMARY KEY,
    station_code VARCHAR(10) NOT NULL UNIQUE,
    latitude DOUBLE,
    longitude DOUBLE
);

CREATE TABLE measurement_variables (
    variable_id INTEGER PRIMARY KEY,
    variable_code VARCHAR(20) NOT NULL UNIQUE,
    label VARCHAR(255) NOT NULL,
    unit VARCHAR(50) NOT NULL
);

CREATE TABLE precipitation_measurements (
    measurement_id INTEGER PRIMARY KEY,
    station_id INTEGER NOT NULL,
    sample_date DATE NOT NULL,
    NS DOUBLE,
    NS_flag DOUBLE,
    LF DOUBLE,
    LF_flag DOUBLE,
    pH DOUBLE,
    pH_flag DOUBLE,
    NH4 DOUBLE,
    NH4_flag DOUBLE,
    Na DOUBLE,
    Na_flag DOUBLE,
    K DOUBLE,
    K_flag DOUBLE,
    Ca DOUBLE,
    Ca_flag DOUBLE,
    Mg DOUBLE,
    Mg_flag DOUBLE,
    Cl DOUBLE,
    Cl_flag DOUBLE,
    NO3 DOUBLE,
    NO3_flag DOUBLE,
    SO4 DOUBLE,
    SO4_flag DOUBLE,
    Pb DOUBLE,
    Pb_flag DOUBLE,
    Cd DOUBLE,
    Cd_flag DOUBLE,
    FOREIGN KEY (station_id) REFERENCES stations(station_id)
);
"""

print(sql_create_statements)


CREATE TABLE stations (
    station_id INTEGER PRIMARY KEY,
    station_code VARCHAR(10) NOT NULL UNIQUE,
    latitude DOUBLE,
    longitude DOUBLE
);

CREATE TABLE measurement_variables (
    variable_id INTEGER PRIMARY KEY,
    variable_code VARCHAR(20) NOT NULL UNIQUE,
    label VARCHAR(255) NOT NULL,
    unit VARCHAR(50) NOT NULL
);

CREATE TABLE precipitation_measurements (
    measurement_id INTEGER PRIMARY KEY,
    station_id INTEGER NOT NULL,
    sample_date DATE NOT NULL,
    NS DOUBLE,
    NS_flag DOUBLE,
    LF DOUBLE,
    LF_flag DOUBLE,
    pH DOUBLE,
    pH_flag DOUBLE,
    NH4 DOUBLE,
    NH4_flag DOUBLE,
    Na DOUBLE,
    Na_flag DOUBLE,
    K DOUBLE,
    K_flag DOUBLE,
    Ca DOUBLE,
    Ca_flag DOUBLE,
    Mg DOUBLE,
    Mg_flag DOUBLE,
    Cl DOUBLE,
    Cl_flag DOUBLE,
    NO3 DOUBLE,
    NO3_flag DOUBLE,
    SO4 DOUBLE,
    SO4_flag DOUBLE,
    Pb DOUBLE,
    Pb_flag DOUBLE,
    Cd DOUBLE,
    Cd_flag DOUBLE,
    FOREIGN KEY (station_id) REFERENCES stations(stati

In [17]:
from pathlib import Path

sql_folder = Path("../sql")
sql_folder.mkdir(exist_ok=True)

sql_file = sql_folder / "create_tables.sql"

with open(sql_file, "w", encoding="utf-8") as f:
    f.write(sql_create_statements)

print("SQL file saved to:", sql_file)

SQL file saved to: ..\sql\create_tables.sql


## 5. Entity Relationship design

The ER diagram documents the final relational structure.

The schema contains:
- one `stations` table,
- one `precipitation_measurements` table,
- one `measurement_variables` metadata table.

The main relationship is:

`stations (1) → (N) precipitation_measurements`

The `measurement_variables` table documents the meaning and units of the measurement columns.

In [19]:
from graphviz import Digraph
from pathlib import Path

output_folder = Path("../outputs")
output_folder.mkdir(exist_ok=True)

er = Digraph("ER_Diagram", format="png")

er.attr(
    rankdir="LR",
    splines="ortho",
    nodesep="1.4",
    ranksep="2.0",
    margin="0.8",
    pad="0.8",
    bgcolor="white"
)

er.attr("node", shape="plain", fontname="Arial")
er.attr("edge", fontname="Arial", fontsize="12", color="black")

er.node("stations", r"""<
<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="9">
  <TR><TD BGCOLOR="#DCEBFF"><B>stations</B></TD></TR>
  <TR><TD ALIGN="LEFT"><B>PK</B> station_id</TD></TR>
  <TR><TD ALIGN="LEFT">station_code</TD></TR>
  <TR><TD ALIGN="LEFT">latitude</TD></TR>
  <TR><TD ALIGN="LEFT">longitude</TD></TR>
</TABLE>
>""")

er.node("precipitation_measurements", r"""<
<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="9">
  <TR><TD BGCOLOR="#DFF5DF"><B>precipitation_measurements</B></TD></TR>
  <TR><TD ALIGN="LEFT"><B>PK</B> measurement_id</TD></TR>
  <TR><TD ALIGN="LEFT"><B>FK</B> station_id</TD></TR>
  <TR><TD ALIGN="LEFT">sample_date</TD></TR>
  <TR><TD ALIGN="LEFT">NS</TD></TR>
  <TR><TD ALIGN="LEFT">NS_flag</TD></TR>
  <TR><TD ALIGN="LEFT">LF</TD></TR>
  <TR><TD ALIGN="LEFT">LF_flag</TD></TR>
  <TR><TD ALIGN="LEFT">pH</TD></TR>
  <TR><TD ALIGN="LEFT">pH_flag</TD></TR>
  <TR><TD ALIGN="LEFT">NH4</TD></TR>
  <TR><TD ALIGN="LEFT">NH4_flag</TD></TR>
  <TR><TD ALIGN="LEFT">Na</TD></TR>
  <TR><TD ALIGN="LEFT">Na_flag</TD></TR>
  <TR><TD ALIGN="LEFT">K</TD></TR>
  <TR><TD ALIGN="LEFT">K_flag</TD></TR>
  <TR><TD ALIGN="LEFT">Ca</TD></TR>
  <TR><TD ALIGN="LEFT">Ca_flag</TD></TR>
  <TR><TD ALIGN="LEFT">Mg</TD></TR>
  <TR><TD ALIGN="LEFT">Mg_flag</TD></TR>
  <TR><TD ALIGN="LEFT">Cl</TD></TR>
  <TR><TD ALIGN="LEFT">Cl_flag</TD></TR>
  <TR><TD ALIGN="LEFT">NO3</TD></TR>
  <TR><TD ALIGN="LEFT">NO3_flag</TD></TR>
  <TR><TD ALIGN="LEFT">SO4</TD></TR>
  <TR><TD ALIGN="LEFT">SO4_flag</TD></TR>
  <TR><TD ALIGN="LEFT">Pb</TD></TR>
  <TR><TD ALIGN="LEFT">Pb_flag</TD></TR>
  <TR><TD ALIGN="LEFT">Cd</TD></TR>
  <TR><TD ALIGN="LEFT">Cd_flag</TD></TR>
</TABLE>
>""")

er.node("measurement_variables", r"""<
<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="9">
  <TR><TD BGCOLOR="#FFF2CC"><B>measurement_variables</B></TD></TR>
  <TR><TD ALIGN="LEFT"><B>PK</B> variable_id</TD></TR>
  <TR><TD ALIGN="LEFT">variable_code</TD></TR>
  <TR><TD ALIGN="LEFT">label</TD></TR>
  <TR><TD ALIGN="LEFT">unit</TD></TR>
</TABLE>
>""")

# Real foreign-key relationship
er.edge(
    "stations",
    "precipitation_measurements",
    label="1 : N",
    arrowhead="crow",
    penwidth="2"
)

# Metadata/documentation relationship, not FK
er.edge(
    "measurement_variables",
    "precipitation_measurements",
    style="dashed",
    arrowhead="none",
    color="gray45",
    penwidth="1.5"
)

output_path = "../outputs/diagrams/er_diagram"
er.render(output_path, cleanup=True)

print("Clean ER diagram saved to:")
print(output_path + ".png")

Clean ER diagram saved to:
../outputs/diagrams/er_diagram.png


## 6. Upload tables to DBRepo

This section prepares each dataframe for DBRepo upload.

For each table, the primary-key column is set as the dataframe index because the DBRepo client uses the index as the table identifier.

Each uploaded table includes a short description containing:
- dataset provenance,
- DOI,
- original creators,
- license information.

In [21]:
SOURCE_METADATA = {
    "original_title": "Concentrations of major ions in wet precipitation samples in Austria",
    "original_creators": "Peter Redl, Thomas Steinkogler, Anne Kasper-Giebl",
    "original_version": "1.0.0",
    "original_doi": "https://doi.org/10.48436/b0g4h-rv840",
    "original_repository": "TU Wien Research Data Repository",
    "original_license": "CC BY-NC-SA 4.0",
    "spatial_coverage": "Austria",
    "temporal_coverage": "2014-10-01 to 2020-12-30",
    "source_files": "precipitationdata.csv; stationcoordinates.csv"
}

### 6.1 Upload `stations`

The `stations` table contains station codes and geographic coordinates.  
The primary key is `station_id`.

In [27]:
stations_table_for_dbrepo = stations_table.copy()

stations_table_for_dbrepo["station_code"] = stations_table_for_dbrepo["station_code"].astype(str)
stations_table_for_dbrepo["latitude"] = stations_table_for_dbrepo["latitude"].astype(float)
stations_table_for_dbrepo["longitude"] = stations_table_for_dbrepo["longitude"].astype(float)

stations_table_for_dbrepo = stations_table_for_dbrepo.set_index("station_id")
stations_table_for_dbrepo.index.name = "station_id"

stations_table_for_dbrepo

,station_code,latitude,longitude
station_id,,,
1,HF,47.470833,10.680833
2,NB,47.662222,12.226944
3,IV,46.818056,12.351667
4,NH,47.956389,13.016667
5,WW,47.421667,13.253333
6,SO,47.054167,12.958889
7,LI,48.955556,15.038889
8,LU,47.855000,15.068611
9,OS,48.220833,15.083889


In [72]:
stations_description = "Stations for Austrian precipitation data derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0."

In [73]:
stations_result = client.create_table(
    database_id=DATABASE_ID,
    name="stations",
    is_public=True,
    is_schema_public=True,
    dataframe=stations_table_for_dbrepo,
    description=stations_description
)

print(stations_result)

2026-05-14 13:22:38,182 root         WARNING default to 'text' for column station_code and type <class 'numpy.dtype'>
id='53688cc1-5205-4f25-af30-7feef2ea1b2b' database_id='bfa4385b-54a9-4ae3-b4f4-cb503d7bb016' name='stations' description='Stations for Austrian precipitation data derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0.' internal_name='stations' is_versioned=True is_public=True is_schema_public=True owned_by='rashidulaminsaad'


### 6.2 Upload `precipitation_measurements`

The `precipitation_measurements` table contains one row per precipitation sample.

The `sample_date` column is converted to text format because the DBRepo Python client handles this format reliably during upload.

The primary key is `measurement_id`.

In [60]:
precipitation_table_for_dbrepo = precipitation_table.copy()

precipitation_table_for_dbrepo["sample_date"] = pd.to_datetime(
    precipitation_table_for_dbrepo["sample_date"]
).dt.strftime("%Y-%m-%d")

precipitation_table_for_dbrepo = precipitation_table_for_dbrepo.set_index("measurement_id")
precipitation_table_for_dbrepo.index.name = "measurement_id"

precipitation_table_for_dbrepo.head()

,station_id,sample_date,NS,NS_flag,LF,LF_flag,pH,pH_flag,NH4,NH4_flag,...,Cl,Cl_flag,NO3,NO3_flag,SO4,SO4_flag,Pb,Pb_flag,Cd,Cd_flag
measurement_id,,,,,,,,,,,,,,,,,,,,,
1,6,2014-10-01,6.700000,1.0,4.05,1.0,5.52,1.0,0.158,1.0,...,0.016,1.0,0.493,1.0,0.573,1.0,0.00017,1.0,0.032874,1.0
2,14,2014-10-01,1.400000,1.0,16.95,1.0,5.72,1.0,1.187,1.0,...,0.105,1.0,4.417,1.0,1.424,1.0,NaN,7.0,NaN,7.0
3,7,2014-10-01,3.800000,1.0,NaN,7.0,NaN,7.0,NaN,7.0,...,NaN,7.0,NaN,7.0,NaN,7.0,NaN,7.0,NaN,7.0
4,8,2014-10-01,11.500000,1.0,13.47,1.0,6.34,1.0,0.381,1.0,...,0.038,1.0,1.788,1.0,0.876,1.0,NaN,7.0,NaN,7.0
5,3,2014-10-01,2.599653,1.0,4.40,1.0,5.77,1.0,0.376,1.0,...,0.050,1.0,0.780,1.0,0.240,1.0,NaN,7.0,NaN,7.0


In [61]:
precipitation_description = "Precipitation chemistry measurements derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0."

In [62]:
precipitation_result = client.create_table(
    database_id=DATABASE_ID,
    name="precipitation_measurements",
    is_public=True,
    is_schema_public=True,
    dataframe=precipitation_table_for_dbrepo,
    description=precipitation_description
)
print("Precipitation measurements table created.")
print(precipitation_result)

2026-05-14 13:10:11,991 root         WARNING default to 'text' for column sample_date and type <class 'numpy.dtype'>
Precipitation measurements table created.
id='d11966e6-f0a6-460d-900b-b56e627fc752' database_id='bfa4385b-54a9-4ae3-b4f4-cb503d7bb016' name='precipitation_measurements' description='Precipitation chemistry measurements derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0.' internal_name='precipitation_measurements' is_versioned=True is_public=True is_schema_public=True owned_by='rashidulaminsaad'


### 6.3 Upload `measurement_variables`

The `measurement_variables` table documents the meaning and units of the chemical measurement variables.

The primary key is `variable_id`.

In [68]:
measurement_variables_for_dbrepo = measurement_variables.copy()

measurement_variables_for_dbrepo = measurement_variables_for_dbrepo.set_index("variable_id")
measurement_variables_for_dbrepo.index.name = "variable_id"

measurement_variables_for_dbrepo

,variable_code,label,unit
variable_id,,,
1,NS,Precipitation amount,mm
2,LF,Conductivity,µS/cm
3,pH,pH value,dimensionless
4,NH4,Ammonium concentration,mg/L
5,Na,Sodium concentration,mg/L
6,K,Potassium concentration,mg/L
7,Ca,Calcium concentration,mg/L
8,Mg,Magnesium concentration,mg/L
9,Cl,Chloride concentration,mg/L


In [69]:
measurement_variables_description = "Measurement variable metadata derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0."

In [70]:
measurement_variables_result = client.create_table(
    database_id=DATABASE_ID,
    name="measurement_variables",
    is_public=True,
    is_schema_public=True,
    dataframe=measurement_variables_for_dbrepo,
    description=measurement_variables_description
)

print("Measurement variables table created.")
print(measurement_variables_result)

2026-05-14 13:18:31,856 root         WARNING default to 'text' for column variable_code and type <class 'numpy.dtype'>
2026-05-14 13:18:31,857 root         WARNING default to 'text' for column label and type <class 'numpy.dtype'>
2026-05-14 13:18:31,857 root         WARNING default to 'text' for column unit and type <class 'numpy.dtype'>
Measurement variables table created.
id='7ed509f5-3356-4318-80b3-c6672b13c4b8' database_id='bfa4385b-54a9-4ae3-b4f4-cb503d7bb016' name='measurement_variables' description='Measurement variable metadata derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0.' internal_name='measurement_variables' is_versioned=True is_public=True is_schema_public=True owned_by='rashidulaminsaad'


## 7. Verify uploaded DBRepo tables

This final check lists the created DBRepo tables and prints their descriptions to confirm that the upload and metadata registration succeeded.

In [74]:
tables = client.get_tables(DATABASE_ID)

for t in tables:
    print(t.name)
    print(t.description)
    print("-----")

stations
Stations for Austrian precipitation data derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0.
-----
measurement_variables
Measurement variable metadata derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0.
-----
precipitation_measurements
Precipitation chemistry measurements derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0.
-----


## Data dictionary

A separate data dictionary is provided in `docs/data_dictionary.md`.

It documents each DBRepo table, each column, the meaning of the variables, units of measurement, and the source of the fields. This supports the WP2 requirement to provide descriptive metadata for the database and tables.

## Provenance documentation

Detailed provenance information is provided in:

`docs/provenance.md`

The provenance document explains:

- original data source,
- original creators,
- transformation workflow,
- relational restructuring,
- generated artefacts,
- DBRepo publication process,
- and reproducibility information.

## 8. Mapping units of measurement

This section describes the semantic annotation of the precipitation dataset by mapping all numeric variables to standardized units using controlled ontologies, and by aligning each variable with a formal unit definition. 

The primary reference ontology is the SI Digital Framework, which provides standardized representations of SI units. 

Each numeric variable physical quantity and associated unit are then mapped to corresponding ontology URIs, with preference given to SI-compliant representations. 


Where necessary, units are normalized into SI-coherent forms:
- mg/L and µg/L converted to kg/m³
- electrical conductivity expressed in siemens per metre
- pH (dimensionless) explicitly marked

The resulting mappings are structured into a metadata payload that preserves both the original unit and its ontological representation. 

This metadata is then uploaded to DBRepo via its REST API


Example ![alt text](mm.png "Title")


In [15]:
# Define a dictionary with the ontological mapping
unit_ontology_map = {
    "mm": "http://si-digital-framework.org/SI/units/millimetre",
    "µS/cm": "http://si-digital-framework.org/SI/units/siemens-per-metre",
    "mg/L": "http://si-digital-framework.org/SI/units/kilogram-per-cubic-metre",
    "µg/L": "http://si-digital-framework.org/SI/units/kilogram-per-cubic-metre",
    "dimensionless": "http://si-digital-framework.org/SI/units/unity"
}


# Mapping the unit to the corresponding ontology entry
measurement_variables["ontology_unit"] = measurement_variables["unit"].map(unit_ontology_map)


'''
measurement_variables = pd.DataFrame([
    {"variable_code": "NS", "label": "Precipitation amount", "unit": "mm"},
    {"variable_code": "LF", "label": "Conductivity", "unit": "µS/cm"},
    {"variable_code": "pH", "label": "pH value", "unit": "dimensionless"},
    {"variable_code": "NH4", "label": "Ammonium concentration", "unit": "mg/L"},
    {"variable_code": "Na", "label": "Sodium concentration", "unit": "mg/L"},
    {"variable_code": "K", "label": "Potassium concentration", "unit": "mg/L"},
    {"variable_code": "Ca", "label": "Calcium concentration", "unit": "mg/L"},
    {"variable_code": "Mg", "label": "Magnesium concentration", "unit": "mg/L"},
    {"variable_code": "Cl", "label": "Chloride concentration", "unit": "mg/L"},
    {"variable_code": "NO3", "label": "Nitrate concentration", "unit": "mg/L"},
    {"variable_code": "SO4", "label": "Sulfate concentration", "unit": "mg/L"},
    {"variable_code": "Pb", "label": "Lead concentration", "unit": "µg/L"},
    {"variable_code": "Cd", "label": "Cadmium concentration", "unit": "µg/L"},
])
'''


'\nmeasurement_variables = pd.DataFrame([\n    {"variable_code": "NS", "label": "Precipitation amount", "unit": "mm"},\n    {"variable_code": "LF", "label": "Conductivity", "unit": "µS/cm"},\n    {"variable_code": "pH", "label": "pH value", "unit": "dimensionless"},\n    {"variable_code": "NH4", "label": "Ammonium concentration", "unit": "mg/L"},\n    {"variable_code": "Na", "label": "Sodium concentration", "unit": "mg/L"},\n    {"variable_code": "K", "label": "Potassium concentration", "unit": "mg/L"},\n    {"variable_code": "Ca", "label": "Calcium concentration", "unit": "mg/L"},\n    {"variable_code": "Mg", "label": "Magnesium concentration", "unit": "mg/L"},\n    {"variable_code": "Cl", "label": "Chloride concentration", "unit": "mg/L"},\n    {"variable_code": "NO3", "label": "Nitrate concentration", "unit": "mg/L"},\n    {"variable_code": "SO4", "label": "Sulfate concentration", "unit": "mg/L"},\n    {"variable_code": "Pb", "label": "Lead concentration", "unit": "µg/L"},\n    {"va

In [16]:
# Creating container for the metadata
metadata_payload = []

for _, row in measurement_variables.iterrows():
    unit = row["unit"]

    ontology_unit = unit_ontology_map.get(unit, None)

    if ontology_unit is None:
        raise ValueError(f"No ontology mapping found for unit: {unit}")

    metadata_payload.append({
        "variable_code": row["variable_code"],
        "label": row["label"],
        "unit": {
            "original": unit,
            "ontology_uri": ontology_unit
        }
    })


# Debugging
print("\n--- Metadata preview ---")
for item in metadata_payload:
    print(item)

'''
# Upload to DBRepo
try:
    print('hello')
    
    response = client.update_metadata(
        database_id=DATABASE_ID,
        metadata=metadata_payload
    )

    print("\nUpload successful!")
    print("Response:", response)
    
except Exception as e:
    print("\nUpload failed!")
    print("Error:", str(e))
'''
    


--- Metadata preview ---
{'variable_code': 'NS', 'label': 'Precipitation amount', 'unit': {'original': 'mm', 'ontology_uri': 'http://si-digital-framework.org/SI/units/millimetre'}}
{'variable_code': 'LF', 'label': 'Conductivity', 'unit': {'original': 'µS/cm', 'ontology_uri': 'http://si-digital-framework.org/SI/units/siemens-per-metre'}}
{'variable_code': 'pH', 'label': 'pH value', 'unit': {'original': 'dimensionless', 'ontology_uri': 'http://si-digital-framework.org/SI/units/unity'}}
{'variable_code': 'NH4', 'label': 'Ammonium concentration', 'unit': {'original': 'mg/L', 'ontology_uri': 'http://si-digital-framework.org/SI/units/kilogram-per-cubic-metre'}}
{'variable_code': 'Na', 'label': 'Sodium concentration', 'unit': {'original': 'mg/L', 'ontology_uri': 'http://si-digital-framework.org/SI/units/kilogram-per-cubic-metre'}}
{'variable_code': 'K', 'label': 'Potassium concentration', 'unit': {'original': 'mg/L', 'ontology_uri': 'http://si-digital-framework.org/SI/units/kilogram-per-cubi

In [19]:
# To check if datase reading from repo works 

'''
tables = client.get_tables(DATABASE_ID)
print(f"Tables in database: {len(tables)}")

for t in tables:
    print(f"- {t.id} | {t.name}")
'''

Tables in database: 3
- 53688cc1-5205-4f25-af30-7feef2ea1b2b | stations
- 7ed509f5-3356-4318-80b3-c6672b13c4b8 | measurement_variables
- d11966e6-f0a6-460d-900b-b56e627fc752 | precipitation_measurements
